# Nemotron **v30 (A)** — rollout generation + scoring (HF, no vLLM) → `rollouts.jsonl`

**Notebook 1 of 2.** Internet is OFF and there's no vLLM wheel, so generation uses **HF `generate`**,
made as fast as possible on one RTX 6000 Pro. Notebook B trains on the JSONL (no gen → fast).

## Speed levers (so it doesn't get stuck like the old code)
- **Unsloth `for_inference`** — ~2× faster decode.
- **`num_return_sequences=G` per prompt** — the G rollouts share one prompt length → batched in a
  single forward, and **Mamba-safe** (no cross-prompt left-pad NaN).
- **Stop-on-`\boxed{}`** — generation halts once all G rollouts close the box, instead of always
  running to `GEN_MAX` (the puzzles box early → big token savings).
- **Incremental + resumable** — each group is appended to `rollouts.jsonl` as it finishes; re-running
  skips already-done prompts (survives the 12 h Kaggle limit / timeouts).
- **Scope it**: default 600 prompts × G=6, `GEN_MAX=1536`, `FOCUS_TYPES` on the weak categories where
  MIXED groups (the RAFT signal) live. Raise only if time allows.

## What it does
Load 30B + **0.85 adapter** (Unsloth) → sample G rollouts/prompt over train.csv → score with the
official `compare_answer` → save groups `{id,type,prompt,answer,rollouts:[{text,boxed,reward}],pass_rate}`.
Prints **pass@1 / pass@G / MIXED%** for reward profiling.

> Realistic budget HF-only: ~600×6 with early-stop ≈ ~1 h. It's slower than vLLM but bounded and
> resumable — and you only generate once, then re-train cheaply in Notebook B.


In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

TRAIN_ON_KAGGLE = 1
USE_PRETRAINED  = 0
assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
print({"TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE, "USE_PRETRAINED": USE_PRETRAINED})

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)
if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]
target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--target", target,
     "--upgrade", "--ignore-installed", wheel],
    check=True,
)
if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)
import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")

In [ ]:
import os, glob

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
MODEL_MAX_LEN = 8192

# 0.85 adapter to sample FROM (warm policy)
SFT_ADAPTER_DIR = "/kaggle/input/models/ramkan07/nemotron-lora-adaptor/pytorch/default/1"

def _find(*pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h: return h[0]
    return ""
RAFT_DATA_PATH = _find("/kaggle/input/competitions/*/train.csv",
                       "/kaggle/input/**/train.csv",
                       r"F:/Hackathons/Kaggle-Nemotron/data_generation/src/train.csv")

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_JSONL = os.path.join(OUT_DIR, "rollouts.jsonl")

SEED = 42

# ── generation scope (HF, no vLLM) ──
N_PROMPTS    = 600        # prompts from train.csv (scope to fit time; raise if budget allows)
G_ROLLOUTS   = 6          # rollouts/prompt via num_return_sequences (one batched forward)
GEN_MAX      = 1536       # hard cap; stop-on-boxed usually ends earlier
GEN_TEMP     = 1.0
GEN_TOP_P    = 1.0
EARLY_STOP   = 1          # 1: stop a batch once all G rollouts close \boxed{} (big speedup)
RESUME       = 1          # 1: skip prompts already in OUT_JSONL (resumable across runs)

# focus on weak categories where MIXED groups (learning signal) live. [] = all.
FOCUS_TYPES  = ["cryptarithm", "equation", "cipher"]   # broad classes from the prompt text

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

SMOKE = 1                 # 1: tiny dry-run (12 prompts, G=4) to confirm gen + scoring work
if SMOKE:
    N_PROMPTS, G_ROLLOUTS = 12, 4

print({"N_PROMPTS": N_PROMPTS, "G": G_ROLLOUTS, "GEN_MAX": GEN_MAX, "early_stop": EARLY_STOP,
       "resume": RESUME, "focus": FOCUS_TYPES, "adapter": os.path.basename(SFT_ADAPTER_DIR),
       "train_csv": bool(RAFT_DATA_PATH), "out": OUT_JSONL, "SMOKE": SMOKE})


In [ ]:
if TRAIN_ON_KAGGLE:
    import glob
    import os
    import subprocess
    import sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    print("Found mamba wheels:", all_mamba)
    print("Found causal-conv1d wheels:", all_causal)

    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA:", torch.version.cuda)

    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime because Nemotron depends on CUDA wheels.")

    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--no-index", "--find-links", packages_dir,
            "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes",
        ],
        check=True,
    )

    def pick_last(wheels):
        return wheels[-1] if wheels else None

    causal_wheel = pick_last(all_causal)
    mamba_wheel = pick_last(all_mamba)
    print("Selected causal wheel:", causal_wheel)
    print("Selected mamba wheel:", mamba_wheel)

    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("Could not find a compatible mamba_ssm wheel under /kaggle/input.")

    print("Offline package installation finished. Restart the kernel if Kaggle keeps stale imports from earlier runs.")
else:
    print("USE_PRETRAINED=1: skipping datasets / trl / mamba_ssm / unsloth installation.")


In [ ]:
# ── load 30B + 0.85 adapter (Unsloth, fast inference) ──
import os, glob, torch
import kagglehub
from unsloth import FastLanguageModel
from peft import PeftModel

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print("model:", MODEL_PATH)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH, max_seq_length=MODEL_MAX_LEN, load_in_4bit=False,
    full_finetuning=False, trust_remote_code=True, attn_implementation="eager", dtype=torch.bfloat16)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # generation; per-prompt G-batch shares length so no actual padding

def _resolve(d):
    if d and os.path.exists(os.path.join(d, "adapter_config.json")): return d
    h = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    return os.path.dirname(sorted(h, key=len)[0]) if h else None
ADAPTER = _resolve(SFT_ADAPTER_DIR)
print("0.85 adapter:", ADAPTER)
if ADAPTER:
    model = PeftModel.from_pretrained(model, ADAPTER, is_trainable=False)
else:
    print("[warn] no 0.85 adapter found -> sampling from the BASE model (weak policy; few correct rollouts).")

FastLanguageModel.for_inference(model)   # ~2x faster decode
model.eval()
print("loaded + for_inference ready.")


In [ ]:
# ── verifier (official metric) + prompt pool from train.csv ──
import re, math
import pandas as pd

SYSTEM_PROMPT = (
    "You solve deterministic logical-puzzle tasks. Infer the exact rule from the examples, apply it "
    "step by step, verify it reproduces the examples, then output the final answer once as "
    "\\boxed{...} with nothing after it.")

def extract_boxed(t):
    if not t: return None
    i = t.rfind("\\boxed{")
    if i == -1:
        m = re.findall(r"\\boxed\{([^{}]*)\}", t); return m[-1].strip() if m else None
    d, j = 1, i + 7
    while j < len(t) and d > 0:
        if t[j] == "{": d += 1
        elif t[j] == "}": d -= 1
        j += 1
    return t[i + 7:j - 1].strip()

def compare_answer(stored, pred):
    if pred is None: return False
    s, p = str(stored).strip(), str(pred).strip()
    if re.fullmatch(r"[01]+", s): return p.lower() == s.lower()
    try: return math.isclose(float(s), float(p), rel_tol=1e-2, abs_tol=1e-5)
    except Exception: return p.lower() == s.lower()

def classify(p):
    pl = p.lower()
    if "transformation rule" in pl or "equation" in pl:
        body = pl.split("examples:")[-1][:140]
        return "cryptarithm" if not any(c.isdigit() for c in body) else "equation"
    if "bit manipulation" in pl or "8-bit" in pl: return "bit_manipulation"
    if "numeral system" in pl: return "numeral"
    if "unit conversion" in pl: return "unit_conversion"
    if "gravit" in pl: return "gravity"
    if "encrypt" in pl or "decrypt" in pl or "cipher" in pl: return "cipher"
    return "other"

df = pd.read_csv(RAFT_DATA_PATH).dropna(subset=["prompt", "answer"]).reset_index(drop=True)
df["_t"] = df["prompt"].map(classify)
if FOCUS_TYPES:
    df = df[df["_t"].isin(FOCUS_TYPES)]
df = df.sample(min(N_PROMPTS, len(df)), random_state=SEED).reset_index(drop=True)
pool = [{"id": str(r["id"]) if "id" in df.columns else str(i), "type": r["_t"],
         "prompt": str(r["prompt"]), "answer": str(r["answer"])} for i, r in df.iterrows()]
print("pool:", len(pool), "| types:", dict(pd.Series([p["type"] for p in pool]).value_counts()))


In [ ]:
# ── generate G rollouts/prompt (HF, stop-on-boxed, resumable) + score + save ──
import json, time, os, torch
from transformers import StoppingCriteria, StoppingCriteriaList

class StopAllBoxed(StoppingCriteria):
    """Stop a batch once EVERY rollout has a closed \\boxed{...}. Checks the last 80 tokens
    every 8 steps -> cheap. Avoids generating the long tail to GEN_MAX after the answer."""
    def __init__(self, tok, plen):
        self.tok = tok; self.plen = plen; self.step = 0
    def __call__(self, ids, scores, **kw):
        self.step += 1
        if self.step % 8 != 0:
            return False
        for row in ids:
            tail = self.tok.decode(row[max(self.plen, row.shape[0] - 80):], skip_special_tokens=True)
            j = tail.rfind("\\boxed{")
            if j < 0:
                return False
            d, k = 1, j + 7
            while k < len(tail) and d > 0:
                if tail[k] == "{": d += 1
                elif tail[k] == "}": d -= 1
                k += 1
            if d != 0:
                return False
        return True

def render(p):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": p + PROMPT_SUFFIX}]
    try:
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    except TypeError:
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

# resume: skip prompts already saved
done = set()
if RESUME and os.path.exists(OUT_JSONL):
    for l in open(OUT_JSONL, encoding="utf-8"):
        try: done.add(json.loads(l)["prompt"])
        except Exception: pass
    print(f"[resume] {len(done)} prompts already in {OUT_JSONL}")
todo = [p for p in pool if p["prompt"] not in done]
print(f"to generate: {len(todo)} / {len(pool)}")

fout = open(OUT_JSONL, "a", encoding="utf-8")
t0 = time.time(); n_hit = 0; n_done = 0
for k, p in enumerate(todo):
    enc = tokenizer(render(p["prompt"]), return_tensors="pt", truncation=True,
                    max_length=MODEL_MAX_LEN - GEN_MAX).to(model.device)
    plen = enc["input_ids"].shape[1]
    sc = StoppingCriteriaList([StopAllBoxed(tokenizer, plen)]) if EARLY_STOP else None
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=GEN_MAX, do_sample=True, temperature=GEN_TEMP,
                             top_p=GEN_TOP_P, num_return_sequences=G_ROLLOUTS,
                             pad_token_id=tokenizer.pad_token_id, stopping_criteria=sc)
    rolls = []
    for s in out:
        txt = tokenizer.decode(s[plen:], skip_special_tokens=True)
        b = extract_boxed(txt)
        rolls.append({"text": txt, "boxed": b, "reward": 1.0 if compare_answer(p["answer"], b) else 0.0})
    nok = sum(r["reward"] > 0 for r in rolls); pr = nok / len(rolls)
    fout.write(json.dumps({**p, "rollouts": rolls, "pass_rate": pr}, ensure_ascii=False) + "\n"); fout.flush()
    n_done += 1; n_hit += int(rolls[0]["reward"] > 0)
    del out, enc; torch.cuda.empty_cache()
    if n_done % 10 == 0 or n_done == len(todo):
        el = (time.time() - t0) / 60
        print(f"[{n_done}/{len(todo)}] {el:.1f}min ~{el/max(1,n_done)*60:.0f}s/prompt "
              f"pass@1~{n_hit/n_done:.0%} last_pr={pr:.0%} type={p['type']}", flush=True)
fout.close()

import numpy as np
G = [json.loads(l) for l in open(OUT_JSONL, encoding="utf-8") if l.strip()]
prs = np.array([g["pass_rate"] for g in G])
print(f"\nDONE: {len(G)} groups in {(time.time()-t0)/60:.1f} min")
print(f"pass@G(any correct)={float(np.mean(prs>0)):.0%} | all-correct={float(np.mean(prs>=1)):.0%} | "
      f"MIXED(learning signal)={float(np.mean((prs>0)&(prs<1))):.0%}")
import pandas as pd
gdf = pd.DataFrame([{"t": g["type"], "mixed": 0 < g["pass_rate"] < 1} for g in G])
print("per-type MIXED %:", gdf.groupby("t")["mixed"].mean().round(2).to_dict())
print(f"-> {OUT_JSONL}  (feed to Notebook B)")
